# Squareprompt Capital ML baseline v4

Each market fits a regularized forecast of next-open to fourth-close return, scaled by volatility known at the decision bar. Features capture momentum at several horizons, trend efficiency, candle shape, opening gaps, volatility and volume. A strongly regularized first fit shrinks forecasts toward zero. With more data, purged chronological folds select the ridge penalty and calibrate forecast strength and error. More recent examples receive greater weight.

The notebook refits after 14 elapsed days when additional training data is available. Training uses up to 2,048 matured examples from the last 90 days or 256 bars, whichever covers more time. Insufficient history means flat exposure. Indicators update every bar. Positions account for pinned fees, spread and adverse funding. Separate entry and exit bands, a sizing deadband, drawdown limits and drift controls limit churn and risk.

Sizing uses recent traded notional, candle trade counts when available, and historical funding to limit liquidity and carry risk. A market factor built from the other selected markets' completed candles estimates common exposure. The exposure stage reduces correlated risk inside each sleeve. The risk stage limits stress losses, pauses after unusually large moves or session losses, and retains a hard drawdown stop.

Every stage receives completed history and returns state. Only `training_history` may be used for fitting, preprocessing or parameter selection. In development it is the historical prefix. In final holdout it is frozen before the embargo. Holdout observations may update causal indicators. Model artifacts and fit events are attached to evaluation outputs. No filesystem, network or future access inside strategy cells.


In [ ]:
import numpy as np

def stage_sources(history, state):
    state["source_count"] = len(state.get("sources", {}))
    interval = state.get("bar_interval_ms", 3600000)
    recent = history.tail(49)
    close = np.asarray(recent["close"], dtype=float)
    volume = np.asarray(recent["volume"], dtype=float)
    notional = close * volume
    state["typical_notional"] = float(np.median(notional))
    activity = np.asarray(recent["trade_count"], dtype=float) if "trade_count" in recent else volume
    if not np.isfinite(activity).all():
        activity = volume
    reference = float(np.median(activity[:-1])) if len(activity) > 1 else float(activity[-1])
    state["activity_ratio"] = float(activity[-1]) / max(0.000001, reference)

    # Funding events in a candle are available only after that candle closes.
    carry_bars = min(len(history), 256, max(1, 24 * 3600000 // interval))
    hours = carry_bars * interval / 3600000
    state["observed_funding_hourly"] = float(history["funding"].tail(carry_bars).sum()) / hours if "funding" in history else 0.0
    state["factor_impulse"] = 0.0
    state["market_beta"] = 0.0
    state["market_correlation"] = 0.0
    if len(recent) < 25:
        return state
    times = np.asarray(recent["time"])
    peers = []
    for market, frame in state.get("cross_asset", {}).items():
        if market == state.get("market"):
            continue
        other = frame.tail(len(recent))
        prices = np.asarray(other["close"], dtype=float)
        if not np.array_equal(times, np.asarray(other["time"])) or not np.isfinite(prices).all() or (prices <= 0).any():
            continue
        peers.append(np.diff(np.log(prices)))
    if peers:
        own = np.diff(np.log(close))
        factor = np.mean(peers, axis=0)
        centered = factor - np.mean(factor)
        covariance = float(centered @ (own - np.mean(own)) / len(own))
        variance = float(np.mean(centered ** 2))
        state["factor_impulse"] = float(np.sum(factor[-12:]) / max(0.000001, np.sqrt(12 * variance)))
        state["market_beta"] = float(np.clip(covariance / max(1e-12, variance), -1.5, 1.5))
        state["market_correlation"] = float(np.clip(covariance / max(1e-12, np.std(own) * np.sqrt(variance)), -1.0, 1.0))
    return state


In [ ]:
import numpy as np

def stage_quality(history, state):
    prices = np.asarray(history[["open", "high", "low", "close"]], dtype=float)
    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError("OHLC prices must be finite and positive")
    if (prices[:, 1] < np.maximum(prices[:, 0], prices[:, 3])).any() or (prices[:, 2] > np.minimum(prices[:, 0], prices[:, 3])).any():
        raise ValueError("Inconsistent OHLC candle")
    if not history["time"].is_monotonic_increasing or history["time"].duplicated().any():
        raise ValueError("Candle times must be unique and increasing")
    volume = np.asarray(history["volume"], dtype=float)
    if not np.isfinite(volume).all() or (volume < 0).any():
        raise ValueError("Volume must be finite and nonnegative")
    interval = state.get("bar_interval_ms", 3600000)
    state["tradable"] = len(history) < 2 or int(history["time"].iloc[-1] - history["time"].iloc[-2]) <= 1.5 * interval
    return state


In [ ]:
def stage_storage(history, state):
    state["as_of"] = int(history["available_at"].iloc[-1])
    state["observed_bars"] = len(history)
    return state


In [ ]:
import numpy as np

def stage_state(history, state):
    returns = np.diff(np.log(np.asarray(history["close"].tail(97), dtype=float)))
    fast = float(np.sqrt(np.mean(returns[-24:] ** 2))) if len(returns) else 0.0
    slow = float(np.sqrt(np.mean(returns ** 2))) if len(returns) else 0.0
    state["volatility"] = max(0.000001, fast, 0.75 * slow)
    state["volatility_ratio"] = fast / max(0.000001, slow)
    interval = state.get("bar_interval_ms", 3600000)
    state["bars_per_year"] = 365.0 * 24 * 3600000 / interval
    return state


In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

model_cache = {}
FEATURE_LOOKBACK = 24
FEATURE_NAMES = ["return_1", "momentum_3", "momentum_8", "momentum_24", "trend_distance",
    "trend_efficiency", "volatility_ratio", "range_ratio", "candle_body", "close_location",
    "volume_surprise", "opening_gap"]

def ml_features(history):
    opening = np.asarray(history["open"], dtype=float)
    high = np.asarray(history["high"], dtype=float)
    low = np.asarray(history["low"], dtype=float)
    close = np.asarray(history["close"], dtype=float)
    log_close = np.log(close)
    returns = np.diff(log_close)
    past = sliding_window_view(returns, FEATURE_LOOKBACK)
    sigma = np.maximum(0.000001, np.sqrt(np.mean(past ** 2, axis=1)))
    recent = past[:, -6:]
    vol_ratio = np.sqrt(np.mean(recent ** 2, axis=1)) / sigma
    start = FEATURE_LOOKBACK
    candle_range = np.maximum(high - low, close * 0.000001)
    ranges = sliding_window_view(candle_range[1:] / close[1:], start)
    volume = np.log1p(np.asarray(history["volume"], dtype=float))
    trend_mean = sliding_window_view(log_close[1:], start).mean(axis=1)
    trend = past[:, -12:]
    momentum = [(log_close[start:] - log_close[start - lag:-lag]) / (sigma * np.sqrt(lag))
                for lag in (1, 3, 8, 24)]
    features = np.column_stack(momentum + [
        (log_close[start:] - trend_mean) / (sigma * np.sqrt(start)),
        trend.sum(axis=1) / np.maximum(0.000001, np.abs(trend).sum(axis=1)),
        vol_ratio, ranges[:, -6:].mean(axis=1) / ranges.mean(axis=1),
        ((close - opening) / candle_range)[start:],
        ((2 * close - high - low) / candle_range)[start:],
        volume[start:] - sliding_window_view(volume[1:], start).mean(axis=1),
        (np.log(opening[start:]) - log_close[start - 1:-1]) / sigma,
    ])
    return features, sigma

def ridge_fit(x, y, alpha):
    recency = np.exp2((np.arange(len(y)) - len(y) + 1) / max(64, len(y) / 2))
    recency /= np.sum(recency)
    mean = recency @ x
    scale = np.maximum(np.sqrt(recency @ (x - mean) ** 2), 0.000001)
    z = np.clip((x - mean) / scale, -5, 5)
    center = recency @ z
    z -= center
    average = float(recency @ y)
    gram = (z.T * recency) @ z
    weights = np.linalg.solve(gram + alpha * np.eye(z.shape[1]), (z.T * recency) @ (y - average))
    intercept = average * len(y) / (len(y) + 64) - float(center @ weights)
    return mean, scale, weights, intercept

def ridge_predict(x, fit):
    mean, scale, weights, intercept = fit
    return np.clip((x - mean) / scale, -5, 5) @ weights + intercept

def select_penalty(x, y, horizon):
    if len(y) < 96:
        return 2.0, 0.5, 1.0
    best = None
    # The last training label must mature before the first validation decision.
    first, second = int(len(y) * 0.60), int(len(y) * 0.80)
    actual = y[first:]
    for alpha in (2.0, 0.5, 0.125):
        predictions = []
        for start, end in ((first, second), (second, len(y))):
            stop = start - horizon
            fit = ridge_fit(x[:stop], y[:stop], alpha)
            predictions.append(ridge_predict(x[start:end], fit))
        predicted = np.concatenate(predictions)
        # Shrink a noisy forecast using a unit-strength prior on the calibration slope.
        gain = float(np.clip((predicted @ actual + 1.0) / (predicted @ predicted + 2.0), 0.0, 1.0))
        error = float(np.sqrt(np.mean((actual - gain * predicted) ** 2)))
        if best is None or error < best[2]:
            best = (alpha, gain, error)
    return best

def stage_models(history, state):
    now = int(state["as_of"])
    horizon = int(state.get("prediction_horizon", 4))
    training = state["training_history"]
    window = int(state.get("training_window", 2048))
    interval = int(state.get("bar_interval_ms", 3600000))
    retrain_ms = int(state.get("retrain_interval_ms") or 0)
    trained_through = int(training["available_at"].iloc[-1]) if len(training) else 0
    refit_due = not model_cache or (retrain_ms > 0 and now - model_cache["fitted_at"] >= retrain_ms)
    state["refit_due"] = refit_due
    if refit_due and len(training) > FEATURE_LOOKBACK + horizon and trained_through > model_cache.get("trained_through", 0):
        span = max(int(state.get("training_days", 90)) * 86400000, 256 * interval)
        training = training[training["available_at"] >= trained_through - span].tail(window + FEATURE_LOOKBACK + horizon)
        if len(training) > FEATURE_LOOKBACK + horizon:
            features, volatility = ml_features(training)
            close = np.asarray(training["close"], dtype=float)
            opening = np.asarray(training["open"], dtype=float)
            # Feature j predicts close[j+horizon] / open[j+1] - 1, using matured outcomes only.
            labels = close[FEATURE_LOOKBACK + horizon:] / opening[FEATURE_LOOKBACK + 1:len(opening) - horizon + 1] - 1
            x = features[:-horizon]
            y = np.clip(labels / (volatility[:-horizon] * np.sqrt(horizon)), -4, 4)
            if len(y) >= int(state.get("minimum_training_rows", 32)):
                alpha, gain, error = select_penalty(x, y, horizon)
                mean, scale, weights, intercept = ridge_fit(x, y, alpha)
                event = {"fitted_at": now, "trained_through": trained_through, "rows": len(y), "alpha": alpha,
                    "mean": mean.tolist(), "scale": scale.tolist(), "weights": weights.tolist(),
                    "intercept": intercept, "forecast_gain": gain, "normalized_error": max(0.5, error)}
                events = model_cache.get("events", []) + [event]
                artifact = {"kind": "volatility_scaled_ridge", "version": "baseline-ml-v4", "features": FEATURE_NAMES,
                    **event, "prediction_horizon": horizon, "fit_events": events, "retrain_interval_ms": retrain_ms,
                    "bar_interval_ms": interval, "feature_clip": 5, "prediction_clip_sigma": 3}
                model_cache.update(event, fit=(mean, scale, weights, intercept), events=events, artifact=artifact,
                    last_feature_time=int(training["time"].iloc[-horizon - 1]))
    state["model_ready"] = bool(model_cache) and len(history) > FEATURE_LOOKBACK
    state["forecast"] = 0.0
    state["forecast_uncertainty"] = 1.0
    if state["model_ready"]:
        features, volatility = ml_features(history.tail(FEATURE_LOOKBACK + 1))
        feature = features[-1]
        horizon_vol = float(volatility[-1]) * np.sqrt(horizon)
        prediction = float(ridge_predict(feature, model_cache["fit"])) * model_cache["forecast_gain"]
        uncertainty = model_cache["normalized_error"] * horizon_vol
        state["forecast"] = float(np.clip(prediction * horizon_vol, -3 * uncertainty, 3 * uncertainty))
        state["forecast_uncertainty"] = float(uncertainty)
        state["feature_outlier_fraction"] = float(np.mean(np.abs((feature - model_cache["mean"]) / model_cache["scale"]) > 4))
        state["model_artifact"] = model_cache["artifact"]
    return state


In [ ]:
def stage_signals(history, state):
    forecast = state.get("forecast", 0.0)
    previous = state.get("previous_position", 0.0)
    costs = state.get("execution_costs", {})
    round_trip = 2 * (costs.get("taker_rate", 0.0) + costs.get("half_spread_rate", 0.0))
    holding_hours = state.get("prediction_horizon", 4) * state.get("bar_interval_ms", 3600000) / 3600000
    funding = costs.get("funding_rate_hourly", 0.0)
    sigma = max(0.000001, state.get("forecast_uncertainty", 1.0))
    side = 1.0 if forecast >= 0 else -1.0
    adverse_funding = max(0.0, side * funding) * holding_hours
    edge = abs(forecast) - adverse_funding
    entry = round_trip + 0.10 * sigma
    state["entry_threshold"] = entry + adverse_funding
    state["target"] = 0.0
    state["confidence"] = 0.0
    if not state.get("model_ready", True) or not state.get("tradable", True):
        return state
    # An open sleeve can stay open below the entry band. Increases still require fresh edge.
    state["hold_position"] = side * previous > 0 and edge > 0.025 * sigma
    if state["hold_position"] or edge > entry:
        state["target"] = side
    if edge > entry:
        state["confidence"] = min(1.0, (edge - entry) / (0.35 * sigma))
    return state


In [ ]:
import math

def stage_sizing(history, state):
    bar_budget = 0.15 / math.sqrt(state.get("bars_per_year", 8760))
    volatility_cap = bar_budget / max(0.000001, state.get("volatility", 0.01))
    weight = min(0.8, volatility_cap)
    # Limit exposure to a small share of the market's usual bar turnover.
    if "typical_notional" in state:
        liquidity_cap = 0.01 * state["typical_notional"] / max(1.0, state.get("capital", 10000.0))
        weight = min(weight, liquidity_cap) * math.sqrt(min(1.0, 2.0 * state.get("activity_ratio", 1.0)))
    side = 1.0 if state["target"] >= 0 else -1.0
    funding = max(0.0, side * state.get("observed_funding_hourly", 0.0))
    hours = state.get("prediction_horizon", 4) * state.get("bar_interval_ms", 3600000) / 3600000
    uncertainty = max(0.000001, state.get("forecast_uncertainty", 1.0))
    excess_carry = max(0.0, funding * hours - 0.025 * uncertainty)
    weight /= 1.0 + excess_carry / (0.10 * uncertainty)
    state["sizing_limit"] = weight

    previous = state.get("previous_position", 0.0)
    allocation = weight * state.get("confidence", 1.0)
    if state.get("hold_position"):
        allocation = min(weight, max(abs(previous), allocation))
    target = state["target"] * allocation
    costs = state.get("execution_costs", {})
    one_way_cost = costs.get("taker_rate", 0.0) + costs.get("half_spread_rate", 0.0)
    deadband = max(0.02, 15.0 * one_way_cost)
    if target * previous > 0 and abs(target - previous) < deadband and abs(previous) <= weight:
        target = previous
    state["target"] = target
    return state


In [ ]:
def stage_exposure(history, state):
    # Reduce common exposure when the peer factor is moving against the sleeve.
    side = 1.0 if state["target"] >= 0 else -1.0
    factor_move = side * state.get("market_beta", 0.0) * state.get("factor_impulse", 0.0)
    headwind = min(2.0, max(0.0, -factor_move - 0.75))
    correlation = abs(state.get("market_correlation", 0.0))
    factor_cap = 1.0 / (1.0 + correlation * headwind)
    cap = min(state.get("sizing_limit", 0.8) * factor_cap, state.get("max_exposure", 1.0))
    state["target"] = max(-cap, min(cap, state["target"]))
    return state


In [ ]:
import numpy as np

risk_memory = {"halted": False, "session": -1, "session_equity": 0.0, "session_halted": False, "cooldown_until": 0}

def stage_risk(history, state):
    drawdown = state["drawdown"]
    now = state.get("as_of", int(history["available_at"].iloc[-1]))
    interval = state.get("bar_interval_ms", 3600000)
    equity = state.get("capital", 1.0)
    session = now // 86400000
    if session != risk_memory["session"]:
        risk_memory.update(session=session, session_equity=equity, session_halted=False)
    if equity <= 0 or drawdown >= 0.15:
        risk_memory["halted"] = True

    close = np.asarray(history["close"].tail(26), dtype=float)
    returns = np.diff(np.log(close))
    if len(returns) >= 8:
        preceding = float(np.sqrt(np.mean(returns[:-1] ** 2)))
        if abs(returns[-1]) > 6 * max(0.000001, preceding):
            risk_memory["cooldown_until"] = now + 3 * interval
    if equity < 0.975 * risk_memory["session_equity"]:
        risk_memory["session_halted"] = True
    if risk_memory["halted"] or risk_memory["session_halted"] or now < risk_memory["cooldown_until"] or not state.get("tradable", True):
        state["target"] = 0.0
    else:
        # Cap the equity loss from a three-sigma move, then taper the cap as losses deepen.
        typical_shock = 3 * max(0.000001, state.get("volatility", 0.01))
        observed_shock = float(np.max(np.abs(returns))) if len(returns) else 0.0
        stress = max(typical_shock, observed_shock)
        drawdown_scale = min(1.0, (0.15 - drawdown) / 0.12)
        stress_budget = max(0.004, 0.55 / np.sqrt(state.get("bars_per_year", 8760)))
        cap = min(state.get("sizing_limit", 0.8), stress_budget / stress) * drawdown_scale
        state["target"] = max(-cap, min(cap, state["target"]))
    state["risk_halted"] = risk_memory["halted"]
    return state


In [ ]:
def stage_retraining(history, state):
    state["training_window"] = 2048
    state["training_days"] = 90
    state["minimum_training_rows"] = 32
    state["prediction_horizon"] = 4
    state["retrain_interval_ms"] = 14 * 86400000
    return state


In [ ]:
def stage_drift(history, state):
    if state.get("volatility_ratio", 1.0) > 2.0 or state.get("feature_outlier_fraction", 0.0) > 0.2:
        state["target"] *= 0.5
    return state


In [ ]:
def stage_registry(history, state):
    state["pipeline_version"] = "baseline-ml-v4"
    return state
